# Gikomba Market Stall Analysis

**Analyst:** Eunice Livingroof | **Cohort:** 1 MILDEVS

Wanjiku runs a fresh-produce stall at Gikomba Market. This notebook answers three
questions: How is the business doing? Is it growing? What should she restock and
where should she focus? Only core Python is used (no pandas or NumPy).

## Dataset

The data is copied exactly as provided: 14 days of sales (KES) and the stock register.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Total, average, highest and lowest sales

**Approach:** `sum`, `len`, `max` and `min` give the totals. `list.index()` finds the
position of the best and worst days. Positions 0 to 6 are week 1, so
`position // 7 + 1` gives the week number.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest = max(daily_sales)
lowest = min(daily_sales)

best_pos = daily_sales.index(highest)
worst_pos = daily_sales.index(lowest)
best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales:   KES {total_sales:,}")
print(f"Average/day:   KES {average_sales:,.2f}")
print(f"Highest day:   {best_day}, week {best_week} (KES {highest:,})")
print(f"Lowest day:    {worst_day}, week {worst_week} (KES {lowest:,})")

Total sales:   KES 46,450
Average/day:   KES 3,317.86
Highest day:   Sat, week 2 (KES 5,600)
Lowest day:    Tue, week 1 (KES 1,950)


## Task 2: Week totals and week-on-week growth

**Approach:** `week_totals()` slices the list into two weeks and returns both totals as
a tuple. We unpack the tuple, then calculate growth as (week 2 − week 1) ÷ week 1.

In [3]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) from a list of 14 daily sales."""
    return sum(sales[:7]), sum(sales[7:14])

week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total: KES {week1_total:,}")
print(f"Week 2 total: KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:+.2f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: +10.66%


## Task 3: Classify every day

**Approach:** `classify_day()` compares a day's sales with the average. More than 20%
above is "Strong", more than 20% below is "Weak", otherwise "Normal". The counts use
the dictionary pattern `counts[label] = counts.get(label, 0) + 1`.

In [4]:
def classify_day(amount, average):
    """Return 'Strong' (>20% above average), 'Weak' (>20% below) or 'Normal'."""
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    return "Normal"

day_labels = []
counts = {}
for number, (day, amount) in enumerate(zip(days, daily_sales), start=1):
    label = classify_day(amount, average_sales)
    day_labels.append(label)
    counts[label] = counts.get(label, 0) + 1
    print(f"Day {number:>2} ({day}): KES {amount:>5,} -> {label}")

print(counts)

Day  1 (Mon): KES 2,300 -> Weak
Day  2 (Tue): KES 1,950 -> Weak
Day  3 (Wed): KES 2,800 -> Normal
Day  4 (Thu): KES 2,100 -> Weak
Day  5 (Fri): KES 3,600 -> Normal
Day  6 (Sat): KES 5,200 -> Strong
Day  7 (Sun): KES 4,100 -> Strong
Day  8 (Mon): KES 2,500 -> Weak
Day  9 (Tue): KES 2,200 -> Weak
Day 10 (Wed): KES 3,100 -> Normal
Day 11 (Thu): KES 2,400 -> Weak
Day 12 (Fri): KES 3,900 -> Normal
Day 13 (Sat): KES 5,600 -> Strong
Day 14 (Sun): KES 4,700 -> Strong
{'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays versus weekends

**Approach:** a loop sends each day's sales into a weekday list (Mon to Fri) or a
weekend list (Sat, Sun). The difference is (weekend average − weekday average) ÷
weekday average.

In [5]:
weekday_sales = []
weekend_sales = []
for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_premium_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekends are {weekend_premium_pct:.1f}% higher than weekdays")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekends are 82.5% higher than weekdays


## Task 5: Reorder list and gross margins

**Approach:** loop over `stock.items()`. A product needs reordering when its quantity
is at or below its reorder level. Gross margin is (price − cost) ÷ price.

In [6]:
def find_reorder_items(stock_register):
    """Return the names of products whose qty is at or below their reorder level."""
    to_reorder = []
    for name, info in stock_register.items():
        if info["qty"] <= info["reorder"]:
            to_reorder.append(name)
    return to_reorder

def gross_margins(stock_register):
    """Return {product: gross margin %} calculated as (price - cost) / price * 100."""
    margins = {}
    for name, info in stock_register.items():
        margins[name] = (info["price"] - info["cost"]) / info["price"] * 100
    return margins

reorder_list = find_reorder_items(stock)
margins = gross_margins(stock)

print("To reorder:", reorder_list)
for name, margin in margins.items():
    print(f"{name:<9} margin {margin:.1f}%")

To reorder: ['Tomatoes', 'Sukuma']
Tomatoes  margin 33.3%
Onions    margin 30.0%
Sukuma    margin 40.0%
Potatoes  margin 33.3%
Avocado   margin 42.9%


## Task 6: Stall Performance Report

**Approach:** f-strings with width and alignment specifiers. `:,` adds thousands
separators and `.2f` gives two decimals.

In [7]:
width = 52
line = "=" * width
print(line)
print("STALL PERFORMANCE REPORT".center(width))
print("Wanjiku's Stall, Gikomba Market (14 days)".center(width))
print(line)
print(f"{'Total sales':<26}KES {total_sales:>12,}")
print(f"{'Average per day':<26}KES {average_sales:>12,.2f}")
print(f"{'Highest day':<26}KES {highest:>12,}  ({best_day}, wk {best_week})")
print(f"{'Lowest day':<26}KES {lowest:>12,}  ({worst_day}, wk {worst_week})")
print("-" * width)
print(f"{'Week 1 total':<26}KES {week1_total:>12,}")
print(f"{'Week 2 total':<26}KES {week2_total:>12,}")
print(f"{'Week-on-week growth':<26}{growth_pct:>+16.2f}%")
print("-" * width)
print(f"{'Weekday average':<26}KES {weekday_avg:>12,.2f}")
print(f"{'Weekend average':<26}KES {weekend_avg:>12,.2f}")
print(f"{'Weekend premium':<26}{weekend_premium_pct:>16.1f}%")
print("-" * width)
print("Day classification")
for label in ("Strong", "Normal", "Weak"):
    print(f"  {label:<24}{counts.get(label, 0):>3} days")
print("-" * width)
print(f"Restock now: {', '.join(reorder_list)}")
print("Gross margin by product")
for name, margin in margins.items():
    print(f"  {name:<24}{margin:>6.1f}%")
print(line)

              STALL PERFORMANCE REPORT              
     Wanjiku's Stall, Gikomba Market (14 days)      
Total sales               KES       46,450
Average per day           KES     3,317.86
Highest day               KES        5,600  (Sat, wk 2)
Lowest day                KES        1,950  (Tue, wk 1)
----------------------------------------------------
Week 1 total              KES       22,050
Week 2 total              KES       24,400
Week-on-week growth                 +10.66%
----------------------------------------------------
Weekday average           KES     2,685.00
Weekend average           KES     4,900.00
Weekend premium                       82.5%
----------------------------------------------------
Day classification
  Strong                    4 days
  Normal                    4 days
  Weak                      6 days
----------------------------------------------------
Restock now: Tomatoes, Sukuma
Gross margin by product
  Tomatoes                  33.3%
  Onions    

## Task 7: Recommendations for Wanjiku

1. **Staff and stock for weekends.** Saturday and Sunday average KES 4,900 against
   KES 2,685 on weekdays, which is 82.5% higher, and all 4 "Strong" days were weekend
   days. Buy extra stock on Friday so the busiest days never run short.
2. **The business is growing.** Week 2 sales were KES 24,400 against KES 22,050 in
   week 1, up 10.7%, and every one of the 7 days sold more than the same day the week
   before.
3. **Restock tomatoes and sukuma now.** Tomatoes (18 kg against a reorder level of
   25 kg) and sukuma (9 kg against 15 kg) are below their limits, and they earn
   33.3% and 40.0% margins.
4. **Fix the quiet weekdays.** 6 of the 14 days were "Weak", all of them Monday to
   Thursday, and weekday sales average only KES 2,685. Try a weekday bundle, for
   example sukuma with onions.
5. **Push avocados.** Avocado has the highest margin (42.9%) and 12 kg in stock, so
   each kilo sold earns more than any other product.